# Enzyme design tutorial — from a theozyme to hit hunting and diversification

We start with the chemistry and nothing else — a handful of catalytic residues arranged around a ligand — and design a scaffold that holds them exactly where they need to be.

**Goal:** starting from a **theozyme**, run the
full enzyme-design workflow and read off the results. In five steps we will:

- inspect what the input actually fixes — coordinates, identity, and the region left to design;
- design a scaffold around the catalytic constellation and score it against the AME criteria;
- carry the conditions from the theozyme onto the chosen design, so the catalytic residues stay
  pinned in the steps that follow;
- redesign the sequence on that design with inverse folding;
- refine it with partial diffusion.

Each section opens with a settings block holding the parameters that matter there. Outputs go to
directories named after those parameters, so sweeping a value keeps the earlier results. See the
[enzyme design guide](../enzyme_design.md) for the full parameter reference.

> Run this notebook with the repository venv. Sampling is submitted to Slurm; the notebook only
> submits and waits.

## Setup

Paths, checkpoint, and Slurm resources. Each path has a default and can be overridden by an
environment variable, so the notebook runs elsewhere without editing.

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*NVML.*")
import os
from pathlib import Path

# Locate the repository by walking up to the `.project-root` marker; $PROJECT_ROOT is the
# fallback when the notebook is run from outside the tree.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".project-root").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / ".project-root").exists():
    _root = os.environ.get("PROJECT_ROOT")
    if not _root:
        raise RuntimeError(
            "Could not find the repository root. Run this notebook from inside the "
            "RFD4-Proteina checkout (it contains a `.project-root` marker), or set the "
            "PROJECT_ROOT environment variable to the checkout path."
        )
    REPO_ROOT = Path(_root)

# Tutorial assets: pre-built theozymes and the helper modules used below.
TUTORIAL_DIR = Path(os.environ.get("ENZYME_TUTORIAL_DIR", REPO_ROOT / "docs/guides/tutorials/enzyme_design"))
OUT_ROOT = Path(os.environ.get("ENZYME_TUTORIAL_RUNS", REPO_ROOT / "runs/enzyme_design_tutorial_v3"))
OUT_ROOT.mkdir(parents=True, exist_ok=True)

CKPT_PATH = os.environ.get(
    "LATENT_CKPT_PATH",
    "/projects/ml/latent_design/latent_data/latent_checkpoints/ldm_checkpoints")
# Always run the repository's current SOTA checkpoint: read it from configs/inference.yaml so
# the tutorial tracks whatever the code ships. Override with $LATENT_CKPT_NAME. The resolved
# name is printed in this block's output below.
def _repo_ckpt_name():
    import re
    inf = REPO_ROOT / "configs/inference.yaml"
    if inf.exists():
        m = re.search(r"^\s*ckpt_name:\s*(\S+)", inf.read_text(), re.M)
        if m:
            return m.group(1)
    return "arch_sweep_2026_07_03_pair_trimult_every2_720k-ema.ckpt"
CKPT_NAME = os.environ.get("LATENT_CKPT_NAME") or _repo_ckpt_name()

PARTITION = os.environ.get("SLURM_PARTITION", "gpu")
GRES      = os.environ.get("SLURM_GRES", "gpu:large:1")
MEM       = "32g"
TIME      = "3:00:00"
NJOBS     = 5      # replicates are split across this many array tasks, one GPU each
OVERWRITE_SAMPLING = False   # False reuses any existing outputs and skips sampling; True re-runs it

# Metric tags, matching the benchmark configuration so the numbers are comparable.
# `design` supplies the clash metric the third pass criterion needs; without it `ame.pass`
# is silently zero.
METRICS_TAGS = "[design,unindexed,3d_coordinate_conditioning,ame_refolding]"
# ─────────────────────────────────────────────────────────────────────────

import re, sys, glob, shutil, time, subprocess
import numpy as np, pandas as pd

WT = str(REPO_ROOT)                     # kept for the Hydra command below
PY = f"{REPO_ROOT}/.venv/bin/python3"   # interpreter for scripts invoked below
sys.path.insert(0, str(TUTORIAL_DIR))
sys.path.insert(0, f"{REPO_ROOT}/src"); sys.path.insert(0, f"{REPO_ROOT}/lib/atomworks/src")
# Subprocesses do not inherit sys.path; pass it through PYTHONPATH.
os.environ["PYTHONPATH"] = f"{REPO_ROOT}/src:{REPO_ROOT}/lib/atomworks/src:" + os.environ.get("PYTHONPATH", "")
os.environ.setdefault("PROJECT_ROOT", str(REPO_ROOT))
os.environ.setdefault("DATA_PATH", "/projects/ml/latent_design/latent_data")
os.environ.setdefault("CCD_MIRROR_PATH", "/projects/ml/frozen_pdb_copies/2026_01_06_ccd")
os.environ.setdefault("PDB_MIRROR_PATH", "/projects/ml/frozen_pdb_copies/2026_01_06_pdb")

def run(cmd, echo=True, env=None):
    e = {**os.environ, **(env or {})}
    p = subprocess.run(cmd, shell=True, cwd=TUTORIAL_DIR, capture_output=True, text=True, env=e)
    if echo and p.stdout.strip(): print(p.stdout.rstrip())
    if p.returncode != 0 and p.stderr.strip(): print("STDERR:", p.stderr[-1200:])
    return p.stdout

def tag(v):
    """Render a parameter value as a filename-safe suffix (1.4 -> 1p4)."""
    return str(v).replace(".", "p").replace("-", "m")


# The injected-noise scale exists once per modality; both are available in every step.
BB_CA_NOISE_KEY = "generation.model.bb_ca.simulation_step_params.sc_scale_noise"
LOCAL_LATENTS_NOISE_KEY = "generation.model.local_latents.simulation_step_params.sc_scale_noise"


def override(ov, name, key, value, label):
    """Append a Hydra override when the value is set, and extend the run name accordingly.

    A value of None leaves the configuration default in place and the name unchanged, so the
    output directory records only what was actually varied.
    """
    if value is None:
        return name
    ov.append(f"{key}={value}")
    return name + f"_{label}{tag(value)}"

STAGE_TIMES = []          # (run_name, seconds) per sampling stage

def launch(out_dir, run_name, inputs, num_replicates, overrides=(), batch_size=8,
           njobs=None, timeout=5400):
    """Submit a sampling run and poll until its metrics land.

    The replicates are spread over `njobs` Slurm array tasks, one GPU each, and merged by the
    concatenation job the launch script queues behind them, so a run finishes in roughly the
    time of `num_replicates / njobs` sequential samples.

    Every call submits; nothing is reused. Output directories carry the parameter values, so
    different settings never collide, while repeating one setting resamples in place.

    Args:
        overrides: extra Hydra overrides carrying this section's settings into the job.
        njobs: number of array tasks; defaults to NJOBS, capped at the replicate count.
    """
    csv = f"{out_dir}/inference_metrics/per_example_metrics.csv"
    if not OVERWRITE_SAMPLING and os.path.exists(csv):
        print(f"[skip] {run_name}: reusing {len(pd.read_csv(csv))} existing designs "
              f"(set OVERWRITE_SAMPLING=True to re-sample)")
        return csv
    t_start = time.time()                       # wall clock: submit -> metrics on disk
    # Clear any previous run for these settings. Without this the poll below would match the
    # stale metrics file and return before the new run has produced anything.
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(f"{out_dir}/inference_samples", exist_ok=True)
    n_tasks = min(njobs or NJOBS, num_replicates)
    cmd = (f'bash "{WT}/scripts/launch_infer.sh" digs '
           f'--njobs {n_tasks} --partition {PARTITION} --gres {GRES} --mem {MEM} --time {TIME} '
           f'--code-dir "{WT}" '
           f'inference_experiment=inference_on_user_inputs '
           f'inputs="{inputs}" num_replicates={num_replicates} batch_size={batch_size} '
           f'metrics_tags={METRICS_TAGS} '
           + " ".join(overrides) + " "
           f'ckpt_path={CKPT_PATH} ckpt_name={CKPT_NAME} '
           f'out_dir="{out_dir}" run_name_={run_name}')
    out = run(cmd, echo=False)                      # the launch script prints a long banner
    jid = re.search(r"Submitted batch job (\d+)", out)
    print(f"[submit] {run_name}: {num_replicates} designs over {n_tasks} tasks"
          + (f" (job {jid.group(1)})" if jid else "")
          + (f"  {' '.join(overrides)}" if overrides else ""))
    deadline = time.time() + timeout
    while time.time() < deadline:
        if os.path.exists(csv):
            dt = time.time() - t_start
            STAGE_TIMES.append((run_name, dt))
            print(f"[done] {run_name}: {len(pd.read_csv(csv))} designs in {dt/60:.1f} min")
            return csv
        time.sleep(60)
    raise TimeoutError(f"{run_name} produced no metrics before the timeout")


print("repository   :", REPO_ROOT)
print("tutorial dir :", TUTORIAL_DIR)
print("run outputs  :", OUT_ROOT)
print("checkpoint   :", CKPT_NAME)

repository   : /net/scratch/yyzhang1/rfd4_share/tutorial
tutorial dir : /net/scratch/yyzhang1/rfd4_share/tutorial/docs/guides/tutorials/enzyme_design
run outputs  : /net/scratch/yyzhang1/rfd4_share/tutorial/runs/enzyme_design_tutorial_v3
checkpoint   : arch_sweep_2026_07_03_pair_trimult_every2_720k-ema.ckpt


### Pass criteria

A design passes (`ame.pass == 1`) when it satisfies all three AME criteria at once:

| Criterion | Threshold |
|---|---|
| `refolded_backbone_rmsd` | ≤ 2.0 Å |
| `refolded_motif_allatom_rmsd` | ≤ 1.5 Å |
| `clash_metrics` protein–ligand | ≤ 0 |

`ame_matrix()` prints one row per design showing the exact values, green where a criterion is
met and red where it is not. Quality metrics follow uncoloured; deviation from the parent
scaffold, where applicable, is grouped separately on a blue background.

In [2]:
AME = {
    ("refolded_backbone_rmsd.rmsd", "<="): 2.0,
    ("refolded_motif_allatom_rmsd.rmsd", "<="): 1.5,
    ("clash_metrics.frac_atoms_clashing_inter_residue_polymer_ligand_1p5A", "<="): 0.0,
}
LABEL = {
    "refolded_backbone_rmsd.rmsd": "refold_bb <=2.0",
    "refolded_motif_allatom_rmsd.rmsd": "refold_motif <=1.5",
    "clash_metrics.frac_atoms_clashing_inter_residue_polymer_ligand_1p5A": "clash_lig <=0",
}

def _ok(v, op, t):
    if pd.isna(v): return None
    return {"<=": v <= t, ">=": v >= t, "<": v < t, ">": v > t, "==": v == t}[op]

def ame_matrix(csv, sort_key="design_unindexed_motif_allatom_rmsd.rmsd",
               context=("design_motif_allatom_rmsd.rmsd", "oracle.overall_plddt"),
               reference=None):
    """Tabulate one row per design, grouped by shading.

    Pass criteria are coloured green or red, quality metrics are left plain, and deviation from
    the parent scaffold is shaded blue. Deviation measures distance from the starting point
    rather than design quality, so it forms its own group and takes no part in the verdict.

    Args:
        reference: parent structure; when given, adds the two deviation columns.
    """
    DEV = ("ca_rmsd_to_parent", "seq_ident_to_parent")
    if reference is not None:
        d = deviation_table(csv, reference).reset_index(drop=True)
    else:
        d = pd.read_csv(csv).reset_index(drop=True)
    # Capture rep and job together: with --njobs>1 each array task restarts replicate
    # numbering, so `rep\d+` alone collides after the tasks are concatenated, and the styled
    # table below sets this as its index (Styler requires a unique index).
    _base = d["example_id"].astype(str).str.extract(r"(rep\d+(?:_job\d+)?|_\d+_job)", expand=False)
    _base = _base.fillna(pd.Series([f"s{i}" for i in range(len(d))], index=d.index))
    _cc = _base.groupby(_base).cumcount()
    d["_id"] = [f"{b}#{k}" if (_base == b).sum() > 1 else b for b, k in zip(_base, _cc)]
    verd = {}
    for (col, op), t in AME.items():
        verd[col] = d[col].map(lambda v: _ok(v, op, t)) if col in d else pd.Series([None]*len(d))
    d["_npass"] = [sum(1 for (c, _op) in AME if verd[c].iloc[i] == True) for i in range(len(d))]
    sk = sort_key if sort_key in d else "design_motif_allatom_rmsd.rmsd"
    order = d.sort_values(["_npass", sk], ascending=[False, True]).index

    for c, _op in AME:                       # a metric absent from this run shows as n/a
        if c not in d:
            d[c] = float("nan")
    ame_cols = [c for (c, _op) in AME]
    qual_cols = [c for c in context if c in d]
    dev_cols = [c for c in DEV if c in d]
    cols = ame_cols + qual_cols + dev_cols
    disp = d.loc[order, ["_id"] + cols + ["_npass"]].copy()
    names = ([LABEL.get(c, c.split(".")[0]) for c in ame_cols + qual_cols]
             + list(dev_cols))
    disp.columns = ["sample"] + names + ["PASS /3"]
    disp = disp.set_index("sample")
    ver_by_disp = {LABEL[c]: [verd[c].loc[i] for i in order] for (c, op), t in AME.items()}
    dev_names = set(names[len(ame_cols) + len(qual_cols):])

    def color(df):
        out = pd.DataFrame("", index=df.index, columns=df.columns)
        for col in df.columns:
            if col == "PASS /3":
                out[col] = "font-weight:bold;background-color:#dfe7ef;color:#123"; continue
            if col in dev_names:                       # deviation from parent: shaded blue
                out[col] = "background-color:#dce9f5;color:#123a5c;border-left:2px solid #6a9fd0"
                continue
            if col not in ver_by_disp: continue        # quality metrics: unshaded
            for r, v in zip(df.index, ver_by_disp[col]):
                out.loc[r, col] = ("background-color:#e6e6e6;color:#555" if v is None
                    else "background-color:#a8dba8;color:#14431a" if v
                    else "background-color:#e79a9a;color:#6a0f0f")
        return out

    def fmt(v):
        if pd.isna(v): return "n/a"
        try: return f"{float(v):.4g}"
        except Exception: return str(v)

    n_pass = int((d["_npass"] == 3).sum())
    print(f"ame.pass {n_pass}/{len(d)} = {n_pass/len(d)*100:.0f}%")
    return disp.style.apply(color, axis=None).format(fmt), d


def describe_conditions(path, title=""):
    """Print what a CIF fixes: conditioned atom counts and the catalytic residues.

    Used on both the theozyme and the design prepared for chaining, so the two are read
    with one implementation and can be compared directly.
    """
    from atomworks.io.utils.annotator import ensure_annotations
    from rfproteina.inference_utils.datasets import DesignProblemDataset

    aa = DesignProblemDataset(dir_path_or_file_paths=str(path), transform=None,
                              num_replicates=1)[0]["atom_array"]
    ensure_annotations(aa, "is_ligand")
    prot = np.asarray(aa.mask("is_polymer"))
    crd, seq, idx = (np.asarray(aa.mask(q)) for q in
                     ("mask_coordinate_1_atom", "mask_sequence_1_residue", "mask_index_1_residue"))
    motif = prot & crd
    if title:
        print(title)
    print(f"  atoms {aa.array_length()}  |  protein {int(prot.sum())}  ligand {int((~prot).sum())}")
    print(f"  C_CRD coordinates fixed: {int(crd.sum())}  "
          f"(protein {int((crd & prot).sum())} + ligand {int((crd & ~prot).sum())})")
    print(f"  C_SEQ identity fixed:    {int(seq.sum())}  "
          f"(protein {int((seq & prot).sum())} + ligand {int((seq & ~prot).sum())})")
    print(f"  designable residues:     {len(set(aa.res_id[prot & idx].tolist()))}")
    for rid in sorted(set(aa.res_id[motif].astype(int))):
        m = motif & (aa.res_id == rid)
        print(f"    res {rid:>4} {aa.res_name[m][0]:<4} {sorted(aa.atom_name[m].tolist())}")
    return aa

### Helpers

These functions are defined here and folded by default; expand a block to read it.

`pick_and_prepare` — selects the top-ranked passing design and carries the theozyme conditions onto it.

In [3]:
def pick_and_prepare(out_dir, verbose=False):
    """Select the top-ranked passing design and carry the theozyme conditions onto it.

    Ranks the run by ``ame.pass``, then by ``design_unindexed_motif_allatom_rmsd`` — a
    deterministic, oracle-free measure of how accurately the motif was placed. The winner is
    copied to ``<out_dir>/best/`` and its ``C_CRD``/``C_SEQ`` are rewritten over the atoms the
    generator marked as coming from the conditioned motif. Returns the prepared CIF path.
    """
    import shutil
    from atomworks.io import parse
    from atomworks.io.utils.annotator import ensure_annotations
    from atomworks.io.utils.io_utils import to_cif_file
    from atomworks.ml.conditions import C_CRD, C_SEQ
    from rfproteina.utils.parse_presets import ANNOTATED_CIF

    PASS, KEY = "ame.pass", "design_unindexed_motif_allatom_rmsd.rmsd"
    best_dir = f"{out_dir}/best"
    dst = f"{best_dir}/best_conditioned.cif"

    ranked = (pd.read_csv(f"{out_dir}/inference_metrics/per_example_metrics.csv")
              .sort_values([PASS, KEY], ascending=[False, True]).reset_index(drop=True))
    passed = ranked[ranked[PASS] == 1]
    if passed.empty:
        raise RuntimeError(f"no design passed ame.pass out of {len(ranked)}; review the run")
    best = passed.iloc[0]
    short = str(best["example_id"]).split("__")[-1].replace("_job0", "")

    # Re-apply the conditions. The generator tagged each output atom that came from a
    # coordinate-conditioned input atom; the boolean of that per-atom column is read back here
    # rather than matching on residue number or atom name, which the new scaffold assigns afresh.
    aa = parse(str(best["designed_structure_path"]), config=ANNOTATED_CIF)["assemblies"]["1"][0]
    ensure_annotations(aa, "is_ligand")
    ensure_annotations(aa, "is_standard_aa")
    mapped_prot = np.asarray(aa.mask("mapped_to_coordinate_conditioned_region & ~hetero"))
    C_CRD.set_mask(aa, mapped_prot | np.asarray(aa.mask("mask_coordinate_1_atom & hetero")))
    C_SEQ.set_mask(aa, mapped_prot | np.asarray(aa.mask("hetero")))
    # C_IDX stays on: the scaffold is a concrete chain now, so the catalytic residues have a
    # definite position and are no longer unindexed.

    import os
    os.makedirs(best_dir, exist_ok=True)
    shutil.copy2(best["designed_structure_path"], f"{best_dir}/best_raw.cif")
    to_cif_file(aa, dst, save_standard_annotations=True, chain_disambiguation="chain_iid")
    ranked.to_csv(f"{best_dir}/selection.csv", index=False)

    if verbose:
        crd, seq = np.asarray(aa.mask("mask_coordinate_1_atom")), np.asarray(aa.mask("mask_sequence_1_residue"))
        print(f"selected {short}: {len(passed)} of {len(ranked)} passed, "
              f"motif placed to {best[KEY]:.3f} A")
        print("conditions re-applied per catalytic residue "
              "(C_CRD fixes coordinates, C_SEQ fixes identity):")
        for rid in sorted(set(aa.res_id[mapped_prot].astype(int))):
            m = mapped_prot & (aa.res_id == rid)
            print(f"  chain {aa.chain_id[m][0]} res {rid:>4} {aa.res_name[m][0]:<4}"
                  f"  C_CRD + C_SEQ on {sorted(aa.atom_name[m].tolist())}")
        print(f"  ligand: C_CRD on {int((crd & np.asarray(aa.mask('hetero'))).sum())} atoms, "
              f"C_SEQ on {int((seq & np.asarray(aa.mask('hetero'))).sum())} atoms")
        print("  wrote:")
        print(f"    {dst}   (conditions re-applied)")
        print(f"    {best_dir}/best_raw.cif   (chosen design, unchanged)")
        print(f"    {best_dir}/selection.csv   (full ranking)")
    return dst

`deviation_table` — adds backbone RMSD and sequence identity of each design relative to the parent scaffold.

In [4]:
from __future__ import annotations
import biotite.structure as struc
from atomworks.io import parse
from atomworks.io.utils.annotator import ensure_annotations
from rfproteina.utils.parse_presets import ANNOTATED_CIF


def _ca_and_seq(path):
    """Return the protein C-alpha coordinates and per-residue names, skipping ligands and gaps."""
    aa = parse(str(path), config=ANNOTATED_CIF)["assemblies"]["1"][0]
    ensure_annotations(aa, "is_ligand")
    sel = aa.mask("(atom_name == 'CA') & ~hetero & ~has_nan_coord()")
    return aa.coord[sel], aa.res_name[sel]


def deviation(generated, reference):
    """``ca_rmsd_to_parent`` and ``seq_ident_to_parent`` for one structure.

    NaN when the residue counts differ, rather than truncating to a common length and comparing
    positions that do not correspond.
    """
    gc, gs = _ca_and_seq(generated)
    rc, rs = _ca_and_seq(reference)
    if len(gc) != len(rc) or len(gc) == 0:
        return {"ca_rmsd_to_parent": float("nan"), "seq_ident_to_parent": float("nan"), "n_res": len(gc)}
    fitted, _ = struc.superimpose(rc, gc)
    rmsd = float(np.sqrt(((fitted - rc) ** 2).sum(-1).mean()))
    ident = float((gs == rs).mean())
    return {"ca_rmsd_to_parent": rmsd, "seq_ident_to_parent": ident, "n_res": len(gc)}


def deviation_table(metrics_csv, reference):
    """Add ``ca_rmsd_to_parent`` and ``seq_ident_to_parent`` columns to a metrics table."""
    d = pd.read_csv(metrics_csv)
    rows = [deviation(p, reference) for p in d["designed_structure_path"]]
    for k in ("ca_rmsd_to_parent", "seq_ident_to_parent"):
        d[k] = [r[k] for r in rows]
    return d

`switchable_view` — a viewer with one button per design and a toggle for all side chains.

In [5]:
from __future__ import annotations
import io, json, uuid
import biotite.structure.io.pdb as biopdb
from IPython.display import HTML

VIEWER_JS = "https://3Dmol.org/build/3Dmol-min.js"


def _to_pdb(sub):
    f = biopdb.PDBFile()
    f.set_structure(sub)
    buf = io.StringIO()
    f.write(buf)
    return buf.getvalue()


def motif_model(cif_path):
    """Write the catalytic residues, whole, as one PDB model.

    Returns ``{"pdb": str, "tips": [[resi, atom_name], ...], "n_rest": int}``. Keeping the
    residues in one model lets bonds be perceived across the tip / non-tip boundary. Tip atoms
    are identified to the viewer by residue number and atom name, carried in the PDB itself.
    """
    aa = parse(str(cif_path), config=ANNOTATED_CIF)["assemblies"]["1"][0]
    ensure_annotations(aa, "is_ligand")
    tip = np.asarray(aa.mask("mapped_to_coordinate_conditioned_region & ~hetero"))

    chain = np.asarray(aa.chain_iid).astype(str)
    keys = set(zip(chain[tip].tolist(), aa.res_id[tip].astype(int).tolist(), strict=False))
    same_res = np.array([(c, int(r)) in keys for c, r in zip(chain, aa.res_id, strict=False)], dtype=bool)
    motif = same_res & ~np.asarray(aa.mask("hetero"))

    tips = [[int(r), str(n)] for r, n in zip(aa.res_id[tip], aa.atom_name[tip], strict=False)]
    return {"pdb": _to_pdb(aa[motif]), "tips": tips, "n_rest": int((motif & ~tip).sum())}


def switchable_view(samples, width=820, height=560):
    """A viewer with one button per sample and a toggle for all side chains.

    ``samples`` are entries of ``{"label", "cif", "motif", "tips", "note"}``, where ``motif`` and
    ``tips`` come from :func:`motif_model`. Returns an ``IPython.display.HTML`` that survives
    being saved into the notebook.
    """
    uid = "v" + uuid.uuid4().hex[:10]

    payload = ",\n".join(
        "{{label:{l},cif:{c},motif:{m},tips:{t},note:{n}}}".format(
            l=json.dumps(s["label"]),
            c=json.dumps(s["cif"]),
            m=json.dumps(s["motif"]),
            t=json.dumps(s["tips"]),
            n=json.dumps(s.get("note", "")),
        )
        for s in samples
    )

    buttons = "".join(
        f'<button id="{uid}_b{i}" onclick="{uid}_show({i})" '
        f'style="margin:2px 4px 2px 0;padding:4px 10px;border:1px solid #bbb;'
        f'border-radius:4px;background:#f4f4f4;cursor:pointer;font:12px monospace">'
        f'{s["label"]}</button>'
        for i, s in enumerate(samples)
    )

    return HTML(f"""
<div style="font:13px/1.5 system-ui,sans-serif">
  <div style="margin-bottom:4px">{buttons}</div>
  <div style="margin-bottom:6px">
    <button id="{uid}_sc" onclick="{uid}_toggle()"
      style="padding:4px 10px;border:1px solid #bbb;border-radius:4px;background:#f4f4f4;
             cursor:pointer;font:12px monospace">show all side chains</button>
  </div>
  <div id="{uid}_note" style="margin:4px 0 8px;color:#333;font:12px monospace"></div>
  <div id="{uid}" style="width:{width}px;height:{height}px;position:relative"></div>
  <div style="margin-top:6px;color:#555;font-size:12px">
    light grey cartoon = scaffold &nbsp;|&nbsp; orange = substrate &nbsp;|&nbsp;
    green = conditioned tip atoms &nbsp;|&nbsp; grey sticks = rest of the catalytic residues
  </div>
</div>
<script src="{VIEWER_JS}"></script>
<script>
(function() {{
  var data = [
{payload}
  ];
  var cur = 0, showAll = false, viewer = null;

  function draw() {{
    var s = data[cur];
    viewer.clear();
    viewer.addModel(s.cif, "cif");                       // model 0: the whole design
    viewer.setStyle({{}}, {{cartoon: {{color: "lightgrey"}}}});
    if (showAll) {{                        // every side chain, thin sticks, elements coloured
      viewer.addStyle({{model: 0, hetflag: false}},
                      {{stick: {{colorscheme: "grayCarbon", radius: 0.08}}}});
    }}
    viewer.setStyle({{hetflag: true}}, {{stick: {{colorscheme: "orangeCarbon", radius: 0.30}}}});

    viewer.addModel(s.motif, "pdb");                     // model 1: catalytic residues, whole
    viewer.setStyle({{model: 1}}, {{stick: {{colorscheme: "grayCarbon", radius: 0.12}}}});
    // Tip atoms are addressed by residue number and atom name rather than by serial, which the
    // viewer assigns independently of the order they were written in.
    for (var t = 0; t < s.tips.length; t++) {{
      var sel = {{model: 1, resi: s.tips[t][0], atom: s.tips[t][1]}};
      viewer.addStyle(sel, {{stick: {{colorscheme: "greenCarbon", radius: 0.24}}}});
      viewer.addStyle(sel, {{sphere: {{scale: 0.32, colorscheme: "greenCarbon"}}}});
    }}

    viewer.zoomTo({{hetflag: true}}); viewer.zoom(0.45); viewer.render();
    document.getElementById("{uid}_note").textContent = s.note;
    for (var i = 0; i < data.length; i++) {{
      var b = document.getElementById("{uid}_b" + i);
      if (b) {{ b.style.background = (i === cur) ? "#cfe3cf" : "#f4f4f4";
                b.style.fontWeight = (i === cur) ? "bold" : "normal"; }}
    }}
    var sc = document.getElementById("{uid}_sc");
    sc.style.background = showAll ? "#cfe3cf" : "#f4f4f4";
    sc.textContent = showAll ? "hide other side chains" : "show all side chains";
  }}

  window["{uid}_show"] = function(i) {{ cur = i; draw(); }};
  window["{uid}_toggle"] = function() {{ showAll = !showAll; draw(); }};

  function build() {{
    var el = document.getElementById("{uid}");
    if (!el || typeof $3Dmol === "undefined") {{ setTimeout(build, 120); return; }}
    viewer = $3Dmol.createViewer(el, {{backgroundColor: "white"}});
    draw();
  }}
  build();
}})();
</script>
""")


def samples_from_metrics(df, n=6, sort_by=None, note_cols=()):
    """Take the first ``n`` designs from a metrics table and prepare them for the viewer."""
    d = df.sort_values(sort_by) if sort_by and sort_by in df else df
    out = []
    for i, (_, row) in enumerate(d.head(n).iterrows()):
        path = row["designed_structure_path"]
        mm = motif_model(path)
        with open(path) as f:
            cif = f.read()
        bits = [
            f"{c.split('.')[0]}={row[c]:.3f}"
            for c in note_cols
            if c in row and isinstance(row[c], int | float) and row[c] == row[c]
        ]
        label = str(row.get("example_id", i)).split("__")[-1].replace("_job0", "")
        out.append({"label": label, "cif": cif, "motif": mm["pdb"], "tips": mm["tips"], "note": "  ".join(bits)})
    return out


def selection_gallery(cif_path, selections, width=820, height=560):
    """Interactive selection viewer: one button per atomworks selection query.

    Each button highlights exactly the atoms ``aa.mask("<query>")`` returns — the mask is evaluated
    here in Python and baked into the viewer as ``(chain, res_id, atom_name)`` triples, so what you
    see is what the selection selects, including coordinate/annotation predicates a viewer's own
    selection language could not express. ``selections`` is a list of ``(label, query)`` pairs.
    """
    aa = parse(str(cif_path), config=ANNOTATED_CIF)["assemblies"]["1"][0]
    ensure_annotations(aa, "is_ligand")
    chain = np.asarray(aa.chain_id).astype(str)
    resi = aa.res_id.astype(int)
    aname = np.asarray(aa.atom_name).astype(str)
    items = []
    for label, q in selections:
        m = np.asarray(aa.mask(q))
        atoms = [[str(chain[i]), int(resi[i]), str(aname[i])] for i in np.where(m)[0]]
        items.append({"label": label, "query": q, "n": int(m.sum()), "atoms": atoms})

    uid = "g" + uuid.uuid4().hex[:10]
    payload = ",\n".join(
        "{{label:{l},query:{q},n:{n},atoms:{a}}}".format(
            l=json.dumps(it["label"]), q=json.dumps(it["query"]),
            n=json.dumps(it["n"]), a=json.dumps(it["atoms"]))
        for it in items)
    buttons = "".join(
        f'<button id="{uid}_b{i}" onclick="{uid}_show({i})" '
        f'style="margin:2px 4px 2px 0;padding:4px 10px;border:1px solid #bbb;border-radius:4px;'
        f'background:#f4f4f4;cursor:pointer;font:12px monospace">{it["label"]}</button>'
        for i, it in enumerate(items))

    return HTML(f"""
<div style="font:13px/1.5 system-ui,sans-serif">
  <div style="margin-bottom:4px">{buttons}</div>
  <div id="{uid}_note" style="margin:4px 0 8px;color:#333;font:12px monospace">&nbsp;</div>
  <div id="{uid}" style="width:{width}px;height:{height}px;position:relative"></div>
  <div style="margin-top:6px;color:#555;font-size:12px">
    grey cartoon = design &nbsp;|&nbsp; green = atoms returned by <code>aa.mask(query)</code>
  </div>
</div>
<script src="{VIEWER_JS}"></script>
<script>
(function() {{
  var data = [
{payload}
  ];
  var pdb = {json.dumps(_to_pdb(aa))};
  var cur = 0, viewer = null;
  function draw() {{
    var s = data[cur];
    viewer.clear();
    viewer.addModel(pdb, "pdb");
    viewer.setStyle({{}}, {{cartoon: {{color: "lightgrey"}}}});
    viewer.setStyle({{hetflag: true}}, {{stick: {{colorscheme: "orangeCarbon", radius: 0.22}}}});
    var ors = [];
    for (var k = 0; k < s.atoms.length; k++)
      ors.push({{chain: s.atoms[k][0], resi: s.atoms[k][1], atom: s.atoms[k][2]}});
    if (ors.length) {{
      viewer.addStyle({{or: ors}}, {{sphere: {{scale: 0.34, colorscheme: "greenCarbon"}}}});
      viewer.addStyle({{or: ors}}, {{stick: {{colorscheme: "greenCarbon", radius: 0.20}}}});
    }}
    viewer.zoomTo(); viewer.render();
    document.getElementById("{uid}_note").innerHTML =
      "aa.mask(<b>" + s.query + "</b>) &rarr; " + s.n + " atoms";
    for (var i = 0; i < data.length; i++) {{
      var b = document.getElementById("{uid}_b" + i);
      if (b) {{ b.style.background = (i === cur) ? "#cfe3cf" : "#f4f4f4";
                b.style.fontWeight = (i === cur) ? "bold" : "normal"; }}
    }}
  }}
  window["{uid}_show"] = function(i) {{ cur = i; draw(); }};
  function build() {{
    var el = document.getElementById("{uid}");
    if (!el || typeof $3Dmol === "undefined") {{ setTimeout(build, 120); return; }}
    viewer = $3Dmol.createViewer(el, {{backgroundColor: "white"}});
    draw();
  }}
  build();
}})();
</script>
""")

# --- one selection-driven RMSD used in section 4 to reproduce the metrics ------------------


def sc_rmsd(gt_coords, pred_coords, sel, group=None):
    """RMSD after superposing ``gt_coords`` onto ``pred_coords`` using the atoms in the boolean
    ``sel`` mask (independently within each ``group`` when given), atom-weighted across groups.
    ``sel`` is what ``aa.mask("<query>")`` returns, so the selection drives everything."""
    from biotite.structure import superimpose, rmsd
    ok = ~np.isnan(gt_coords).any(1) & ~np.isnan(pred_coords).any(1)
    groups = np.zeros(len(sel), dtype=int) if group is None else group
    w = t = 0.0
    for g in np.unique(groups):
        a = (groups == g) & sel & ok
        n = int(a.sum())
        if n == 0:
            continue
        fitted, _ = superimpose(gt_coords[a], pred_coords[a])
        w += float(rmsd(gt_coords[a], fitted)) ** 2 * n
        t += n
    return float((w / t) ** 0.5)


## 1. Inspect the input

Before designing anything, it pays to read exactly what the input has locked down and what it has left open.

The theozyme fixes the catalytic geometry and leaves everything else to the model: each
catalytic residue keeps only the atoms named by the benchmark, the ligand is pinned in part,
the catalytic residues are **unindexed** so the model chooses where they sit in the chain, and
an **expandable segment** stands in for the scaffold to be designed.

The tutorial ships one worked example: `M0732_1xs1`, an MCSA benchmark entry whose theozyme carries the catalytic residues and the ligand with the benchmark's own conditions already applied. Point `THEOZYME` at your own annotated CIF to run the same workflow on a different constellation; the [annotated CIF tutorial](annotated_cif_tutorial.ipynb) covers building one.

In [6]:
# ── Settings — input ─────────────────────────────────────────────
THEOZYME = TUTORIAL_DIR / "M0732_1xs1_theozyme.cif"   # or your own annotated CIF
# ─────────────────────────────────────────────────────────────────

from pathlib import Path
theo = Path(THEOZYME)
if not theo.exists():
    raise FileNotFoundError(f"{theo} does not exist.")

print("theozyme:", theo)

theozyme: /net/scratch/yyzhang1/rfd4_share/tutorial/docs/guides/tutorials/enzyme_design/M0732_1xs1_theozyme.cif


Read back what the input fixes: coordinates (`C_CRD`), identity (`C_SEQ`), and the size of the region left to design.

In [7]:
aa = describe_conditions(theo, "theozyme")
motif = aa.mask("mask_coordinate_1_atom & ~hetero")

theozyme
  atoms 224  |  protein 196  ligand 28
  C_CRD coordinates fixed: 28  (protein 16 + ligand 12)
  C_SEQ identity fixed:    44  (protein 16 + ligand 28)
  designable residues:     180
    res  180 ALA  ['C', 'O']
    res  181 ARG  ['C', 'O']
    res  182 GLU  ['CD', 'CG', 'OE1', 'OE2']
    res  183 SER  ['C', 'CA', 'CB', 'N']
    res  184 ARG  ['CZ', 'NE', 'NH1', 'NH2']


### The catalytic constellation

Green marks the coordinate-conditioned motif atoms, orange the substrate. There is no scaffold
yet — this is all the model is given.

In [8]:
import py3Dmol, biotite.structure.io.pdb as biopdb, io as _io

def atomarray_to_pdb(sub):
    f = biopdb.PDBFile(); f.set_structure(sub)
    buf = _io.StringIO(); f.write(buf); return buf.getvalue()

show = (motif | aa.mask("hetero")) & aa.mask("~has_nan_coord()")   # skip the placeholder
v = py3Dmol.view(width=720, height=520)
v.addModel(atomarray_to_pdb(aa[show]), "pdb")
v.setStyle({"stick": {"colorscheme": "greenCarbon", "radius": 0.18}})
v.setStyle({"hetflag": True}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.22}})
v.addStyle({"not": {"hetflag": True}}, {"sphere": {"scale": 0.25}})
v.setBackgroundColor("0xffffff"); v.zoomTo()
print("green: conditioned motif atoms   orange: substrate")
v.show()

green: conditioned motif atoms   orange: substrate


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 2. Scaffold design

Sampling starts from pure noise: backbone and sequence are generated together, constrained
throughout by the motif geometry. This is the de novo step. It returned the lowest pass rate of
the three sampling steps in the runs recorded here, on one catalytic site under one
checkpoint.

### Settings — scaffold generation

| Setting | Config key | Default | Effect |
|---|---|---|---|
| `NUM_REPLICATES` | `num_replicates` | 1 | Scaffolds to sample. Only a fraction pass, so sample more than you need |
| `BATCH_SIZE` | `batch_size` | 1 | Designs denoised together on one GPU. A step costs nearly the same whatever the batch holds, so keep batches full |
| `NSTEPS` | `nsteps` | 100 | Denoising steps |
| `GUIDANCE_W` | `guidance_w` | 1.0 | 1.0 turns guidance off. Any other value enables classifier-free guidance, which evaluates the network a second time per step and roughly doubles the sampling time |
| `BB_CA_SC_SCALE_NOISE` | `bb_ca.sc_scale_noise` | 0.3 | Injected-noise scale for the backbone |
| `LOCAL_LATENTS_SC_SCALE_NOISE` | `local_latents.sc_scale_noise` | 0.1 | Injected-noise scale for the sequence |

`None` leaves the configuration default in place. Every setting here except `NUM_REPLICATES` and
`BATCH_SIZE` also applies to steps 4 and 5, where it appears again with a step-specific name.

In [9]:
# ── Settings ─────────────────────────────────────────────────────
NUM_REPLICATES              = 25    # num_replicates: scaffolds to sample
BATCH_SIZE                  = 5     # batch_size: designs denoised together on one GPU
NSTEPS                      = None  # generation.args.nsteps; None keeps the default, 100
GUIDANCE_W                  = None  # generation.args.guidance_w; None keeps the default, 1.0
BB_CA_SC_SCALE_NOISE        = None  # bb_ca sc_scale_noise; None keeps the default, 0.3
LOCAL_LATENTS_SC_SCALE_NOISE = None # local_latents sc_scale_noise; None keeps the default, 0.1
# ─────────────────────────────────────────────────────────────────

We are now ready to submit the batch and see how many designs land the motif accurately enough to pass.

In [10]:
ov2, name2 = [], f"n{NUM_REPLICATES}"
name2 = override(ov2, name2, "generation.args.nsteps", NSTEPS, "s")
name2 = override(ov2, name2, "generation.args.guidance_w", GUIDANCE_W, "g")
name2 = override(ov2, name2, BB_CA_NOISE_KEY, BB_CA_SC_SCALE_NOISE, "bb")
name2 = override(ov2, name2, LOCAL_LATENTS_NOISE_KEY, LOCAL_LATENTS_SC_SCALE_NOISE, "ll")

OUT2 = str(OUT_ROOT / f"scaffold_{name2}")
csv2 = launch(OUT2, "s2design", theo, NUM_REPLICATES, ov2, BATCH_SIZE)

[skip] s2design: reusing 25 existing designs (set OVERWRITE_SAMPLING=True to re-sample)


### Results

One row per design. `design_unindexed_motif_allatom_rmsd` measures how accurately the motif was
placed and is computed without the refolding oracle, so it is deterministic; the refolded
columns come from AF3 and carry the pass criteria.

In [11]:
styled2, df2 = ame_matrix(csv2)
styled2

ame.pass 3/25 = 12%


,refold_bb <=2.0,refold_motif <=1.5,clash_lig <=0,design_motif_allatom_rmsd,oracle,PASS /3
sample,,,,,,
rep1_job4,0.5223,1.221,0,0.783,84.76,3
rep1_job1,0.6963,1.279,0,0.8184,93.28,3
rep1_job0,0.8173,1.324,0,0.8891,89.24,3
rep0_job0,0.4489,2.194,0,0.8086,90.55,2
rep2_job1,1.22,3.333,0,0.7654,89.35,2
rep4_job1,0.6575,1.854,0,0.9788,91.52,2
rep3_job3,0.9699,2.946,0,0.8068,93.52,2
rep0_job3,0.431,2.09,0,1.048,93.94,2
rep4_job4,0.9315,2.282,0,0.8997,92.56,2


## 3. Carry the conditions onto the chosen design

A generated CIF records where the motif went but does not carry the conditions forward: on the
output, `C_CRD` and `C_SEQ` cover the ligand only. Fed onward unchanged, inverse folding would
redesign the catalytic residues and partial diffusion would let them drift. This step re-applies
the conditions, and so stands between scaffolding and the two refinement steps.

The correspondence is recorded per atom. As the generator inserts the unindexed motif into the
scaffold, each output atom that came from a coordinate-conditioned input atom is tagged in the
CIF column `_atom_site.annotation_src-coordinate-conditioned-region_1_atom`, whose value points
back to the source catalytic residue. `pick_and_prepare` (in the setup helpers above) reads the
boolean form of that column — `mapped_to_coordinate_conditioned_region` — and rewrites `C_CRD`
and `C_SEQ` over exactly those atoms plus the ligand. The selection follows the atoms the
generator marked, not residue numbers or atom names, which the new scaffold assigns afresh.

In [12]:
best_cif = pick_and_prepare(OUT2, verbose=True)

selected M0732_1xs1_theozyme_rep1_job4: 3 of 25 passed, motif placed to 0.413 A
conditions re-applied per catalytic residue (C_CRD fixes coordinates, C_SEQ fixes identity):
  chain A res   92 SER   C_CRD + C_SEQ on ['C', 'CA', 'CB', 'N']
  chain A res   96 ARG   C_CRD + C_SEQ on ['CZ', 'NE', 'NH1', 'NH2']
  chain A res  121 ALA   C_CRD + C_SEQ on ['C', 'O']
  chain A res  123 ARG   C_CRD + C_SEQ on ['C', 'O']
  chain A res  160 GLU   C_CRD + C_SEQ on ['CD', 'CG', 'OE1', 'OE2']
  ligand: C_CRD on 12 atoms, C_SEQ on 28 atoms
  wrote:
    /net/scratch/yyzhang1/rfd4_share/tutorial/runs/enzyme_design_tutorial_v3/scaffold_n25/best/best_conditioned.cif   (conditions re-applied)
    /net/scratch/yyzhang1/rfd4_share/tutorial/runs/enzyme_design_tutorial_v3/scaffold_n25/best/best_raw.cif   (chosen design, unchanged)
    /net/scratch/yyzhang1/rfd4_share/tutorial/runs/enzyme_design_tutorial_v3/scaffold_n25/best/selection.csv   (full ranking)


### Verify before chaining

Read the prepared CIF with the same function used on the theozyme in section 1. The conditioned
atom counts must match: the catalytic geometry that was pinned going in is the geometry pinned
going on. A mismatch here means the next two steps would optimise the wrong thing.

In [13]:
_ = describe_conditions(best_cif, "prepared for chaining")

prepared for chaining
  atoms 1525  |  protein 1497  ligand 28
  C_CRD coordinates fixed: 28  (protein 16 + ligand 12)
  C_SEQ identity fixed:    44  (protein 16 + ligand 28)
  designable residues:     180
    res   92 SER  ['C', 'CA', 'CB', 'N']
    res   96 ARG  ['CZ', 'NE', 'NH1', 'NH2']
    res  121 ALA  ['C', 'O']
    res  123 ARG  ['C', 'O']
    res  160 GLU  ['CD', 'CG', 'OE1', 'OE2']


### Inspect the designs

Use the buttons to switch between designs. Catalytic residues are drawn in full: the
coordinate-conditioned tip atoms in green, the rest of each residue in grey, which shows at a
glance how much of the side chain the conditioning actually pins. Substrate in orange, scaffold
as a grey cartoon.

In [14]:
switchable_view(samples_from_metrics(
    df2, n=6, sort_by="design_unindexed_motif_allatom_rmsd.rmsd",
    note_cols=("design_unindexed_motif_allatom_rmsd.rmsd",
               "refolded_motif_allatom_rmsd.rmsd", "oracle.overall_plddt")))

## 4. Query the design with the selection language

Every output CIF carries the model's per-atom annotations, so you can pull out any region with the
**[atomworks selection syntax](https://rosettacommons.github.io/atomworks/latest/io/utils/selection_syntax.html)** —
`aa.mask("<query>")` returns a boolean mask over the atoms. Pick a query with the buttons below to see
exactly what it matches on the chosen design: the mask is evaluated in Python and baked into the viewer,
so what you see is what the selection selects — including annotation and coordinate predicates
(`mapped_to_coordinate_conditioned_region`, `z > …`) that a viewer's own selection language could not
express. Drag to rotate, scroll to zoom.

For more on working with these files directly, see the atomworks guides on
[annotating and saving structures](https://rosettacommons.github.io/atomworks/latest/auto_examples/annotate_and_save_structures.html)
and [loading and visualizing structures](https://rosettacommons.github.io/atomworks/latest/auto_examples/load_and_visualize_structures.html).

In [15]:
from atomworks.ml.conditions.annotator import ensure_annotations

# The design chosen in section 3 (top-ranked by ame.pass, then motif RMSD) -- same rule as
# pick_and_prepare -- loaded from its run path, whether it was just sampled or reused.
chosen = (pd.read_csv(csv2)
          .sort_values(["ame.pass", "design_unindexed_motif_allatom_rmsd.rmsd"], ascending=[False, True])
          .iloc[0])
chosen_cif = chosen["designed_structure_path"]
_aa = parse(chosen_cif, config=ANNOTATED_CIF)["assemblies"]["1"][0]
_zmid = float(np.median(_aa.coord[:, 2]))

selection_gallery(chosen_cif, [
    ("motif tip atoms",     "mapped_to_coordinate_conditioned_region & ~hetero"),
    ("substrate",           "hetero"),
    ("backbone C\u03b1",   "atom_name == 'CA'"),
    ("aromatics",           "res_name in ['PHE','TYR','TRP','HIS']"),
    (f"z > {_zmid:.0f}",    f"z > {_zmid:.0f}"),
    ("sulfurs",             "element == 'S'"),
])

The gallery highlights a selection visually; the *same* query language also **logs which
design residue mapped onto which input-motif position**. The motif selection and the
per-residue filter are both `aa.mask(...)`; the source label (`A1`…`A5`, one per theozyme
catalytic residue) is a data column, read with `get_annotation`.

### Let's reproduce the metrics with the selection language!

Want to select the motif atoms yourself — to recompute a metric, or to do your own magic? Here's
a showcase. The same `aa.mask(...)` selections rebuild the pipeline's own scores: below, the
**motif all-atom RMSD** (`refolded_motif_allatom_rmsd`) and the **backbone RMSD**
(`refolded_backbone_rmsd`) for the chosen design are recomputed from the CIFs by superposition,
and land on the values already in the metrics table.

In [16]:
design = parse(chosen["designed_structure_path"], config=ANNOTATED_CIF)["assemblies"]["1"][0]
refold = parse(chosen["oracle.refolded_structure_path"], config=ANNOTATED_CIF)["assemblies"]["1"][0]
for _s in (design, refold):
    ensure_annotations(_s, "is_protein_backbone")


Before the RMSDs, read the conditioning back off the loaded design — **which design residue
was mapped onto which input-motif position**. The motif selection and the per-residue filter
are both `aa.mask(...)`; the source label (`A1`…`A5`, one per theozyme catalytic residue) is a
data column, read with `get_annotation`.

In [17]:
from rfproteina.utils.standard_annotations import SA_CRDSRC

# The motif selection uses aa.mask(...); the input-motif position is a per-atom data column.
motif = design.mask("mapped_to_coordinate_conditioned_region & ~hetero")
src   = np.asarray(SA_CRDSRC.annotation(design))
resids = sorted(set(design.res_id[motif].astype(int).tolist()))
print(f"catalytic motif: {int(motif.sum())} atoms over {len(resids)} residues")
for rid in resids:
    m = motif & (design.res_id == rid)
    print(f"  design res {rid:>4} {design.res_name[m][0]:<4}  <-  input motif {src[m][0]}"
          f"   {sorted(design.atom_name[m].tolist())}")
print(f"  substrate: {int(design.mask('hetero').sum())} ligand atoms")

catalytic motif: 16 atoms over 5 residues
  design res   92 SER   <-  input motif A4   ['C', 'CA', 'CB', 'N']
  design res   96 ARG   <-  input motif A5   ['CZ', 'NE', 'NH1', 'NH2']
  design res  121 ALA   <-  input motif A1   ['C', 'O']
  design res  123 ARG   <-  input motif A2   ['C', 'O']
  design res  160 GLU   <-  input motif A3   ['CD', 'CG', 'OE1', 'OE2']
  substrate: 28 ligand atoms


In [18]:
backbone = sc_rmsd(design.coord, refold.coord,
                   design.mask("is_protein_backbone"))
print(f"backbone RMSD         reproduced {backbone:.4f}   reported {chosen['refolded_backbone_rmsd.rmsd']:.4f}")

backbone RMSD         reproduced 0.5223   reported 0.5223


In [19]:
from rfproteina.utils.standard_annotations import SA_CWI, SA_CRDGRPSRC

sel       = design.mask("mapped_to_coordinate_conditioned_region & ~hetero")  # catalytic motif atoms (query)
motif_xyz = SA_CWI.annotation(design)          # input-motif geometry (data column)
groups    = SA_CRDGRPSRC.annotation(design)    # per-atom Kabsch group (data column)

motif = sc_rmsd(motif_xyz, refold.coord, sel, group=groups)
print(f"motif all-atom RMSD   reproduced {motif:.4f}   reported {chosen['refolded_motif_allatom_rmsd.rmsd']:.4f}")

motif all-atom RMSD   reproduced 1.2211   reported 1.2211


## 5. Inverse folding

The backbone is held fixed and resampled to the input at every step; only the sequence and side
chains are drawn afresh. This is a special case of partial diffusion and cannot be combined with
its parameters.

### Settings — sequence sampling noise scale

| Setting | Config key | Default | Effect |
|---|---|---|---|
| `IF_LOCAL_LATENTS_SC_SCALE_NOISE` | `local_latents.sc_scale_noise` | 0.1 | Injected-noise scale for the sequence. Higher values give more varied sequences |
| `IF_NUM_REPLICATES` | `num_replicates` | 1 | Sequences to sample |
| `IF_BB_CA_SC_SCALE_NOISE` | `bb_ca.sc_scale_noise` | 0.3 | Injected-noise scale for the backbone |
| `IF_NSTEPS` | `nsteps` | 100 | Denoising steps |
| `IF_GUIDANCE_W` | `guidance_w` | 1.0 | 1.0 turns guidance off |

The step is selected by the `generation=inverse_folding` preset, applied in the execution cell
below; `generation.args.inverse_folding=true` does the same thing.

We use 1.0 for the sequence noise scale to broaden diversity. Diversity trades against the pass
rate of the individual sequences, and 1.0 to 1.5 is a reasonable starting point for enzyme
design; tune it when the results fall short.

In [20]:
# ── Settings ─────────────────────────────────────────────────────
IF_LOCAL_LATENTS_SC_SCALE_NOISE = 1.0   # local_latents sc_scale_noise; default 0.1
IF_NUM_REPLICATES               = 10    # num_replicates: sequences to sample
IF_BB_CA_SC_SCALE_NOISE         = None  # bb_ca sc_scale_noise; None keeps the default, 0.3
IF_NSTEPS                       = None  # generation.args.nsteps; None keeps the default, 100
IF_GUIDANCE_W                   = None  # generation.args.guidance_w; None keeps the default, 1.0
# ─────────────────────────────────────────────────────────────────

In [21]:
ov3, name3 = ["generation=inverse_folding"], f"n{IF_NUM_REPLICATES}"
name3 = override(ov3, name3, LOCAL_LATENTS_NOISE_KEY, IF_LOCAL_LATENTS_SC_SCALE_NOISE, "ll")
name3 = override(ov3, name3, BB_CA_NOISE_KEY, IF_BB_CA_SC_SCALE_NOISE, "bb")
name3 = override(ov3, name3, "generation.args.nsteps", IF_NSTEPS, "s")
name3 = override(ov3, name3, "generation.args.guidance_w", IF_GUIDANCE_W, "g")

OUT3 = str(OUT_ROOT / f"inverse_folding_{name3}")
# Two tasks of five sequences each fill BATCH_SIZE exactly.
csv3 = launch(OUT3, "s3invfold", best_cif, IF_NUM_REPLICATES, ov3, BATCH_SIZE, njobs=2)

[skip] s3invfold: reusing 10 existing designs (set OVERWRITE_SAMPLING=True to re-sample)


### Results

Two further columns report how far each design has moved from the parent scaffold:
`ca_rmsd_to_parent` over Cα coordinates and `seq_ident_to_parent` over the sequence. Inverse
folding holds the backbone, so the first stays at zero and the whole change appears in the second.

In [22]:
styled3, df3 = ame_matrix(csv3, reference=best_cif)
styled3

ame.pass 4/10 = 40%


,refold_bb <=2.0,refold_motif <=1.5,clash_lig <=0,design_motif_allatom_rmsd,oracle,ca_rmsd_to_parent,seq_ident_to_parent,PASS /3
sample,,,,,,,,
rep4_job1,0.7878,1.486,0,0.1148,89.93,0.0004967,0.7833,3
rep4_job0,0.5877,1.423,0,0.1221,87.59,0.0004991,0.6889,3
rep2_job0,0.494,1.428,0,0.1248,86.47,0.0005028,0.7944,3
rep0_job0,0.4947,1.364,0,0.157,90.13,0.0004966,0.7778,3
rep3_job1,0.5381,1.582,0,0.094,91.67,0.0005121,0.8444,2
rep2_job1,0.5251,2.045,0,0.1117,90.1,0.0004868,0.7722,2
rep3_job0,0.4738,1.917,0,0.1337,85.2,0.0004836,0.7722,2
rep1_job0,0.4755,1.804,0,0.1397,90.84,0.0004909,0.7889,2
rep1_job1,0.743,1.796,0,0.1606,85.06,0.0004993,0.7611,2


### Inspect the designs

In [23]:
switchable_view(samples_from_metrics(
    df3, n=6, sort_by="refolded_motif_allatom_rmsd.rmsd",
    note_cols=("refolded_motif_allatom_rmsd.rmsd", "ca_rmsd", "seq_ident", "oracle.overall_plddt")))

## 6. Partial diffusion

The design is re-noised to a point part-way along the trajectory and diffused back. Unlike
inverse folding, both modalities are resampled, so the backbone moves as well. Under the default settings the sequence is redrawn more freely while the backbone only takes a small nudge — a local search around an existing solution rather than a pure redesign.

### Settings — how much to change

`PARTIAL_FRACTION` is a fraction in (0, 1]. It re-runs the final part of the diffusion
trajectory: the chosen design is corrupted to a partly-noised state and resampled from there, so
larger values re-run more of the trajectory and move the design further. At 1 the whole
trajectory is re-run and the enzyme scaffold is resampled from scratch, as in a de novo design.
Because the backbone (`bb_ca`) is sampled slightly ahead of the latent that carries the sequence
and side chains (`local_latents`), the sequence tends to change more noticeably while the
backbone shifts by a small amount. At the fraction of 0.5 used in this tutorial, the
design as a whole shifts by a relatively small nudge from the input scaffold.

| Setting | Config key | Default | Effect |
|---|---|---|---|
| `PARTIAL_FRACTION` | `partial_trajectory_fraction_steps` | `null` | Fraction of the trajectory to re-run, in (0, 1]. Larger values change the design more |
| `PD_NUM_REPLICATES` | `num_replicates` | 1 | Refined variants to generate |
| `PD_BB_CA_SC_SCALE_NOISE` | `bb_ca.sc_scale_noise` | 0.3 | Injected-noise scale for the backbone |
| `PD_LOCAL_LATENTS_SC_SCALE_NOISE` | `local_latents.sc_scale_noise` | 0.1 | Injected-noise scale for the sequence |
| `PD_NSTEPS` | `nsteps` | 100 | Denoising steps |
| `PD_GUIDANCE_W` | `guidance_w` | 1.0 | 1.0 turns guidance off |

Concretely, `PARTIAL_FRACTION` sets the number of steps to run — `round(f * nsteps)` — and the
noise level is read from the schedule at that index. It indexes steps, not noise: the `bb_ca`
and `local_latents` schedules are nonlinear, so a small fraction leaves few steps to denoise.

Each modality's starting point can instead be set directly with
`generation.args.partial_t_backbone` and `partial_t_latent`, both in (0, 1), which lets the
backbone be redrawn further than the sequence or the reverse. Their direction is the opposite of
`PARTIAL_FRACTION`: a value near 0 runs most of the trajectory and changes the design more. Give
both, or give one and set `generation.args.couple_partial_t=true` to derive the other from the
schedule. They are mutually exclusive with `PARTIAL_FRACTION`.

In [24]:
# ── Settings ─────────────────────────────────────────────────────
PARTIAL_FRACTION                = 0.5   # generation.args.partial_trajectory_fraction_steps, (0, 1]
PD_NUM_REPLICATES               = 10    # num_replicates: refined variants to generate
PD_BB_CA_SC_SCALE_NOISE         = None  # bb_ca sc_scale_noise; None keeps the default, 0.3
PD_LOCAL_LATENTS_SC_SCALE_NOISE = None  # local_latents sc_scale_noise; None keeps the default, 0.1
PD_NSTEPS                       = None  # generation.args.nsteps; None keeps the default, 100
PD_GUIDANCE_W                   = None  # generation.args.guidance_w; None keeps the default, 1.0
# ─────────────────────────────────────────────────────────────────

In [25]:
ov4 = [f"generation.args.partial_trajectory_fraction_steps={PARTIAL_FRACTION}"]
name4 = f"frac{tag(PARTIAL_FRACTION)}_n{PD_NUM_REPLICATES}"
name4 = override(ov4, name4, BB_CA_NOISE_KEY, PD_BB_CA_SC_SCALE_NOISE, "bb")
name4 = override(ov4, name4, LOCAL_LATENTS_NOISE_KEY, PD_LOCAL_LATENTS_SC_SCALE_NOISE, "ll")
name4 = override(ov4, name4, "generation.args.nsteps", PD_NSTEPS, "s")
name4 = override(ov4, name4, "generation.args.guidance_w", PD_GUIDANCE_W, "g")

OUT4 = str(OUT_ROOT / f"partial_diffusion_{name4}")
# Two tasks of five variants each fill BATCH_SIZE exactly.
csv4 = launch(OUT4, "s4partial", best_cif, PD_NUM_REPLICATES, ov4, BATCH_SIZE, njobs=2)

[skip] s4partial: reusing 10 existing designs (set OVERWRITE_SAMPLING=True to re-sample)


### Results

The same deviation columns apply. Both modalities are resampled here, so `ca_rmsd_to_parent`
is clearly non-zero: the backbone has moved, which is what separates this step from inverse folding.

In [26]:
styled4, df4 = ame_matrix(csv4, reference=best_cif)
styled4

ame.pass 3/10 = 30%


,refold_bb <=2.0,refold_motif <=1.5,clash_lig <=0,design_motif_allatom_rmsd,oracle,ca_rmsd_to_parent,seq_ident_to_parent,PASS /3
sample,,,,,,,,
rep2_job1,0.3698,1.078,0,0.1833,89.84,0.4006,0.7778,3
rep0_job0,0.4021,1.374,0,0.2066,86.16,0.3001,0.6944,3
rep1_job1,0.4949,1.286,0,0.2237,93.33,0.4747,0.7278,3
rep3_job1,0.6249,3.152,0,0.1715,90.24,0.4252,0.8056,2
rep0_job1,0.4039,2.528,0,0.1925,89.38,0.4178,0.6444,2
rep1_job0,0.5118,1.502,0,0.1996,85.46,0.3349,0.8278,2
rep4_job0,1.895,2.258,0,0.2104,85.46,0.3404,0.7667,2
rep4_job1,0.3903,1.639,0,0.2136,91.04,0.3757,0.7722,2
rep2_job0,0.4049,2.348,0,0.219,86.88,0.4175,0.7111,2


### Inspect the designs

In [27]:
switchable_view(samples_from_metrics(
    df4, n=6, sort_by="refolded_motif_allatom_rmsd.rmsd",
    note_cols=("refolded_motif_allatom_rmsd.rmsd", "ca_rmsd", "seq_ident", "oracle.overall_plddt")))

## Summary

| Step | Starts from | Backbone | Sequence | What selects it |
|---|---|---|---|---|
| 2. Scaffold design | noise | generated | generated | neither switch below |
| 4. Inverse folding | the design from step 3 | fixed | resampled | `generation.args.inverse_folding=true` |
| 5. Partial diffusion | the design from step 3 | perturbed | perturbed | `generation.args.partial_trajectory_fraction_steps` |

The two switches are mutually exclusive; setting both raises an error.

Every other parameter applies to all three steps: `num_replicates`, `batch_size`,
`generation.args.nsteps`, `generation.args.guidance_w`, and the injected-noise scale of each
modality — `generation.model.bb_ca.simulation_step_params.sc_scale_noise`, default 0.3, for the
backbone, and the same key under `local_latents`, default 0.1, for the sequence. This tutorial
varies only the sequence one, in step 4; the backbone scale is left at its default throughout.

Steps 4 and 5 both start from the design prepared in step 3, so their pass rates are not
comparable with that of step 2.

To change a parameter, edit the settings block and re-run from that section. Output directories
carry the parameter values, so earlier results are kept.